# 11. Improving the AUC with a proper final test set

Two goals in one notebook:
1. **Use the error analysis (notebook 09) to build better features and test them.**
2. **Do it with a clean protocol**: a final test set is split off first and **not touched until the last section**. Every choice (features, tuning, blending) is made with cross-validation on the remaining 80% ("dev").

## What the error analysis told us, and what we try
| Finding in notebook 09 | Idea |
|---|---|
| Missed defaulters have high external scores, and the model leans on `EXT_SOURCE_MEAN` | **B: more external-score features**: max, std, a weighted mean, pairwise differences, interactions with age, employment and loan burden |
| `INST_DAYS_LATE_max` (lifetime) is a top feature; weak groups include people with little history | **C: recent payment behaviour**: lateness, underpayment and payment ratio in the **last 365 days** only |
| Caught and wrongly flagged applicants look alike; one model may be limiting | **Blend**: average the ranks of LightGBM and XGBoost (different models make different mistakes) |

**Quick version** (short on time): 3-fold CV instead of 5, no Optuna re-tune, XGBoost blend only (no logistic regression).

**Decision rules, fixed before running:**
- A feature group is kept only if it improves AUC in **all 3 folds** (paired, same folds).
- A blend is used only if its out-of-fold AUC beats LightGBM alone by **more than 0.001**.
- Realistic expectation: **+0.002 to +0.008** AUC. Top Kaggle solutions needed far more work for about 0.80.

In [1]:
# Load data, build the three feature sets (A base, B + external-score features, C + recent payments)
import sys, json, time, gc
sys.path.append("..")

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import optuna
from pathlib import Path
from scipy.stats import rankdata
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.features import clean, add_features, add_ext_features, one_hot

optuna.logging.set_verbosity(optuna.logging.WARNING)

df = pd.read_csv("../data/raw/HC_application_train.csv")
y = df["TARGET"].reset_index(drop=True)
app0 = add_features(clean(df.drop(columns="TARGET")))
app = add_ext_features(app0)
ext_cols = [c for c in app.columns if c not in app0.columns]
for name in ["bureau", "previous", "installments"]:
    app = app.join(pd.read_pickle(f"../data/processed/{name}.pkl"), on="SK_ID_CURR")
rec = pd.read_pickle("../data/processed/installments_recent.pkl")
rec_cols = list(rec.columns)
app = app.join(rec, on="SK_ID_CURR")
X_all = app.drop(columns="SK_ID_CURR").reset_index(drop=True)
for c in X_all.select_dtypes(include=["object", "string"]):
    X_all[c] = X_all[c].astype("category")
base_cols = [c for c in X_all.columns if c not in ext_cols + rec_cols]
sets = {"A": base_cols, "B": base_cols + ext_cols, "C": base_cols + ext_cols + rec_cols}
print({k: len(v) for k, v in sets.items()}, "| new external-score features:", len(ext_cols), "| recent-payment features:", len(rec_cols))
del df, app, app0; gc.collect()

{'A': 518, 'B': 528, 'C': 534} | new external-score features: 10 | recent-payment features: 6


20

**Insight:** set A has 518 features, B adds 10 external-score features (528) and C adds 6 recent-payment features (534).

## Split off the final test set first
20% of applicants, stratified, fixed seed. It is stored in `X_hold` / `y_hold` and **not used again until the final section**. Everything else is "dev".

In [2]:
# Lock 20% of applicants as the final test set; everything else is 'dev' for all decisions
idx_dev, idx_hold = train_test_split(np.arange(len(X_all)), test_size=0.2, stratify=y, random_state=2024)
np.save("../models/test_idx.npy", idx_hold)

X_dev = X_all.iloc[idx_dev].reset_index(drop=True); y_dev = y.iloc[idx_dev].reset_index(drop=True)
X_hold = X_all.iloc[idx_hold].reset_index(drop=True); y_hold = y.iloc[idx_hold].reset_index(drop=True)   # DO NOT USE before the final section
del X_all; gc.collect()
print("dev:", X_dev.shape, f"{y_dev.mean():.2%} positives | final test (locked):", X_hold.shape, f"{y_hold.mean():.2%} positives")

folds = list(StratifiedKFold(3, shuffle=True, random_state=42).split(X_dev, y_dev))
base_params = {k: v for k, v in json.load(open("../models/best_params.json")).items() if k != "n_estimators"}

def lgb_params(params, lr):
    return {**params, "learning_rate": lr, "objective": "binary", "metric": "auc", "verbosity": -1, "seed": 42, "num_threads": 6}

def lgb_cv(cols, params, lr=0.05):
    aucs, rounds, oof = [], [], np.zeros(len(X_dev))
    for tr, va in folds:
        dt = lgb.Dataset(X_dev.iloc[tr][cols], y_dev.iloc[tr])
        dv = lgb.Dataset(X_dev.iloc[va][cols], y_dev.iloc[va], reference=dt)
        m = lgb.train(lgb_params(params, lr), dt, 5000, valid_sets=[dv], callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = m.predict(X_dev.iloc[va][cols], num_iteration=m.best_iteration)
        aucs.append(roc_auc_score(y_dev.iloc[va], oof[va])); rounds.append(m.best_iteration)
    return np.array(aucs), int(np.mean(rounds)), oof

dev: (246008, 534) 8.07% positives | final test (locked): (61503, 534) 8.07% positives


**Insight:** dev has 246,008 applicants and the locked test set 61,503; both keep 8.07% positives, so the stratified split preserved the class balance.

## 1. Do the new feature groups help? (3-fold CV on dev, same folds)
The hyper-parameters come from notebook 06. They were tuned on all rows, **including the rows now locked as the test set**, so absolute test AUC may be slightly optimistic. Baseline and improved models use the same settings, so the comparison between them is still fair. A strict version would re-tune on dev only (skipped to save time).

In [3]:
# 3-fold CV on dev for each feature set, same folds, to see whether the new features help
t0 = time.time()
cv = {}
for name, cols in sets.items():
    cv[name] = lgb_cv(cols, base_params)[0]
    print(name, len(cols), "features:", cv[name].mean().round(4), "folds:", cv[name].round(4), f"({time.time()-t0:.0f}s)", flush=True)

tbl = pd.DataFrame({"features": {k: len(v) for k, v in sets.items()},
                    "AUC mean": {k: v.mean() for k, v in cv.items()},
                    "AUC std": {k: v.std() for k, v in cv.items()}})
tbl["paired gain vs A"] = [(cv[k] - cv["A"]).mean() for k in tbl.index]
tbl["folds better than A"] = [int((cv[k] > cv["A"]).sum()) for k in tbl.index]
tbl.round(4)

A 518 features: 0.7863 folds: [0.7917 0.7865 0.7808] (131s)


B 528 features: 0.7861 folds: [0.7921 0.7849 0.7812] (268s)


C 534 features: 0.787 folds: [0.7933 0.7857 0.7821] (413s)


,features,AUC mean,AUC std,paired gain vs A,folds better than A
A,518,0.7863,0.0044,0.0000,0
B,528,0.7861,0.0045,-0.0003,2
C,534,0.7870,0.0047,0.0007,2


**Insight:** the mean AUC is 0.7863 for A, 0.7861 for B (-0.0003) and 0.7870 for C (+0.0007). Neither new group beats A in more than 2 of the 3 folds, and the differences are far smaller than the fold std of about 0.0045, so there is no evidence that the new features help.

In [4]:
# Pre-agreed rule: keep a feature group only if it beats the previous set in every fold
chosen_name, chosen_cols = "A", sets["A"]
for name in ["B", "C"]:
    if (cv[name] > cv[chosen_name]).sum() == len(folds):
        chosen_name, chosen_cols = name, sets[name]
print("Feature set chosen by the all-folds rule:", chosen_name, f"({len(chosen_cols)} features)")

Feature set chosen by the all-folds rule: A (518 features)


**Insight:** because neither group improved every fold, the pre-agreed rule keeps the original 518 features. The new features are dropped, not because they hurt, but because they gave no measurable gain.

## 2. Final LightGBM settings
Quick version: no re-tune, the notebook 06 settings are reused (see the note above).

In [5]:
# Quick version: reuse the notebook 06 settings instead of re-tuning
params_dev = base_params

In [6]:
# CV with the final settings (lower learning rate); gives out-of-fold predictions for blending
t0 = time.time()
aucs_lgb, rounds_lgb, oof_lgb = lgb_cv(chosen_cols, params_dev, lr=0.03)
print("LightGBM, 3-fold CV on dev:", aucs_lgb.mean().round(4), "+/-", aucs_lgb.std().round(4),
      "| trees:", rounds_lgb, f"({time.time()-t0:.0f}s)")

LightGBM, 3-fold CV on dev: 0.7872 +/- 0.0046 | trees: 990 (254s)


**Insight:** with the final settings LightGBM reaches 0.7872 (std 0.0046) using 990 trees on dev, in line with the 0.7863 at the faster learning rate; the difference is noise.

## 3. Does XGBoost add something? (blend, out-of-fold on dev)
XGBoost uses fixed, conservative settings (not tuned), scored on the same three folds.

In [7]:
# XGBoost with fixed conservative settings, out-of-fold predictions on the same folds
t0 = time.time()
xgb_params = {"objective": "binary:logistic", "eval_metric": "auc", "tree_method": "hist", "learning_rate": 0.05, "seed": 42,
              "max_depth": 4, "min_child_weight": 50, "colsample_bytree": 0.4, "subsample": 0.8,
              "reg_lambda": 5, "reg_alpha": 1, "max_cat_to_onehot": 1, "nthread": 6}
oof_xgb, xgb_rounds = np.zeros(len(X_dev)), []
for tr, va in folds:
    dt = xgb.DMatrix(X_dev.iloc[tr][chosen_cols], y_dev.iloc[tr], enable_categorical=True)
    dv = xgb.DMatrix(X_dev.iloc[va][chosen_cols], y_dev.iloc[va], enable_categorical=True)
    m = xgb.train(xgb_params, dt, 5000, evals=[(dv, "va")], early_stopping_rounds=100, verbose_eval=False)
    oof_xgb[va] = m.predict(dv, iteration_range=(0, m.best_iteration + 1)); xgb_rounds.append(m.best_iteration + 1)
print("XGBoost OOF AUC:", round(roc_auc_score(y_dev, oof_xgb), 4), "rounds:", int(np.mean(xgb_rounds)), f"({time.time()-t0:.0f}s)")

XGBoost OOF AUC: 0.7862 rounds: 826 (313s)


**Insight:** XGBoost with fixed conservative settings reaches an out-of-fold AUC of 0.7862 after 826 rounds, within 0.001 of LightGBM, so the two are effectively tied.

In [8]:
# Blend by averaging ranks; use the blend only if it beats LightGBM alone by more than 0.001
rk = lambda v: rankdata(v) / len(v)   # rank-normalise so models on different scales can be averaged
oofs = {"LightGBM": oof_lgb, "XGBoost": oof_xgb}
combos = {"LightGBM alone": ["LightGBM"], "LightGBM + XGBoost": ["LightGBM", "XGBoost"]}
blend = pd.DataFrame({"out-of-fold AUC": {k: roc_auc_score(y_dev, np.mean([rk(oofs[m]) for m in v], axis=0)) for k, v in combos.items()}})
blend["gain vs LightGBM alone"] = blend["out-of-fold AUC"] - blend.loc["LightGBM alone", "out-of-fold AUC"]
display(blend.round(4))

best_combo = blend["out-of-fold AUC"].idxmax()
use = combos[best_combo] if blend.loc[best_combo, "gain vs LightGBM alone"] > 0.001 else ["LightGBM"]
print("Models used in the final prediction (rule: blend only if gain > 0.001):", use)

,out-of-fold AUC,gain vs LightGBM alone
LightGBM alone,0.7871,0.0000
LightGBM + XGBoost,0.7878,0.0006


Models used in the final prediction (rule: blend only if gain > 0.001): ['LightGBM']


**Insight:** averaging LightGBM and XGBoost adds 0.0006 AUC, below the 0.001 threshold set in advance, so LightGBM is used alone. The small gain is probably because both are gradient-boosted trees on the same features and make similar mistakes.

## 4. Final test: locked 20%, scored once
Everything above was decided on dev. Now the chosen pipeline is fitted on all dev rows and scored on the locked test set. The baseline (notebook 06 features and settings) is scored as a reference only. No decision is made after this point.

In [9]:
# Fit on all dev rows, score the locked test set once, and bootstrap the uncertainty
n_lgb = int(rounds_lgb * 1.1)                       # a little more data than in CV, so slightly more trees
final_lgb = lgb.train(lgb_params(params_dev, 0.03), lgb.Dataset(X_dev[chosen_cols], y_dev), n_lgb)
parts = {"LightGBM": final_lgb.predict(X_hold[chosen_cols])}
if "XGBoost" in use:
    dt = xgb.DMatrix(X_dev[chosen_cols], y_dev, enable_categorical=True)
    fx = xgb.train(xgb_params, dt, int(np.mean(xgb_rounds) * 1.1))
    parts["XGBoost"] = fx.predict(xgb.DMatrix(X_hold[chosen_cols], enable_categorical=True))
pred_new = np.mean([rk(parts[m]) for m in use], axis=0)

n_base = json.load(open("../models/best_params.json"))["n_estimators"]
base_model = lgb.train(lgb_params(base_params, 0.03), lgb.Dataset(X_dev[sets["A"]], y_dev), n_base)
pred_base = base_model.predict(X_hold[sets["A"]])

auc_new, auc_base = roc_auc_score(y_hold, pred_new), roc_auc_score(y_hold, pred_base)
rng = np.random.default_rng(0)   # bootstrap: resample test rows 300 times to get a 95% interval
yh = y_hold.to_numpy()
diffs, news = [], []
for _ in range(300):
    i = rng.integers(0, len(yh), len(yh))
    a, b = roc_auc_score(yh[i], pred_new[i]), roc_auc_score(yh[i], pred_base[i])
    news.append(a); diffs.append(a - b)
ci = lambda v: tuple(np.percentile(v, [2.5, 97.5]).round(4))
final = pd.DataFrame({"AUC on locked test": [auc_base, auc_new]}, index=["baseline (notebook 06 setup)", f"improved ({chosen_name}, {' + '.join(use)})"]).round(4)
display(final)
print(f"improved - baseline = {auc_new - auc_base:+.4f}  (95% bootstrap interval {ci(diffs)})")
print(f"improved AUC 95% bootstrap interval: {ci(news)}")
json.dump({"feature_set": chosen_name, "models": use, "params": params_dev, "trees": n_lgb,
           "test_auc_improved": round(auc_new, 4), "test_auc_baseline": round(auc_base, 4),
           "gain": round(auc_new - auc_base, 4), "gain_ci95": ci(diffs)}, open("../models/final_test_result.json", "w"), indent=2)

,AUC on locked test
baseline (notebook 06 setup),0.7922
"improved (A, LightGBM)",0.7922


improved - baseline = -0.0000  (95% bootstrap interval (np.float64(-0.0002), np.float64(0.0002)))
improved AUC 95% bootstrap interval: (np.float64(0.786), np.float64(0.7979))


**Insight:** on the locked test set both models score 0.7922 (difference -0.0000, 95% interval -0.0002 to 0.0002). They are almost the same model, because the rules kept the original features and settings; only the tree count differs. The 95% bootstrap interval for the test AUC is 0.786 to 0.798, close to the cross-validated 0.7914 from notebook 06. The settings were originally tuned on all rows, so a strictly clean estimate would re-tune on dev only.

In [10]:
# Which features did the final model use most (share of total gain)?
gain = pd.Series(final_lgb.feature_importance("gain"), index=chosen_cols)
new_feats = [c for c in chosen_cols if c in ext_cols + rec_cols]
print("Top 12 features by gain in the final LightGBM:")
display((gain / gain.sum()).sort_values(ascending=False).head(12).round(4))
print("New features, rank among", len(chosen_cols), "(1 = most important):")
rank = gain.rank(ascending=False).astype(int)
display(rank[new_feats].sort_values())

Top 12 features by gain in the final LightGBM:


EXT_SOURCE_MEAN                       0.1994
ORGANIZATION_TYPE                     0.0695
EXT_SOURCE_MIN                        0.0485
EXT_SOURCE_3                          0.0329
ANNUITY_CREDIT_RATIO                  0.0226
EXT_SOURCE_PROD                       0.0216
CREDIT_GOODS_RATIO                    0.0204
EXT_SOURCE_2                          0.0173
OCCUPATION_TYPE                       0.0169
EXT_SOURCE_1                          0.0118
PREV_DAYS_LAST_DUE_1ST_VERSION_max    0.0114
AMT_ANNUITY                           0.0110
dtype: float64

New features, rank among 518 (1 = most important):


Series([], dtype: int64)

**Insight:** `EXT_SOURCE_MEAN` carries about 20% of the total gain, far ahead of `ORGANIZATION_TYPE` (7%) and `EXT_SOURCE_MIN` (5%). The list of new features is empty because set A won, so none of them are in the final model.

## Findings
- **The error-analysis ideas did not raise the AUC.** Extra external-score features changed the CV AUC by -0.0003 and recent-payment features by +0.0007, both far below the fold spread (about 0.0045), so neither passed the all-folds rule.
- **Blending did not pass either:** LightGBM + XGBoost gained 0.0006, under the 0.001 threshold set in advance.
- **Honest final number:** AUC on the locked 20% test set is **0.7922** (95% bootstrap interval 0.786 to 0.798). The improved and baseline models are effectively the same model, so there is no measured improvement over the notebook 06 setup.
- **Reading it:** the model seems to be near what these tables can give. The missed defaulters in notebook 09 look like good applicants on the available data, so more features built from the same tables are unlikely to separate them. Gains would need new information or much heavier feature work, and the next realistic step is tuning XGBoost and others properly, not more features from these tables.
- **Caveats:** quick version (3 folds, no re-tune on dev only, no logistic regression in the blend); the notebook 06 settings had seen the locked rows, so the 0.7922 could be slightly optimistic.